# 🧮 Step 4: SOFA Score Calculation & Feature Engineering

## What This Notebook Does

Transforms the cleaned continuous-time data from Step 2 into **model-ready arrays**
that the Bayesian HMM in Step 5 can directly consume.

### What is SOFA?
**SOFA** (Sequential Organ Failure Assessment) is a clinical scoring system that measures
dysfunction across **6 organ systems**, each scored 0-4:

| Organ System | What It Measures | Key Variable | Score Range |
|-------------|-----------------|--------------|-------------|
| **Respiratory** | Lung function | PaO2/FiO2 ratio or SpO2 | 0-4 |
| **Coagulation** | Blood clotting | Platelet count | 0-4 |
| **Liver** | Liver function | Bilirubin | 0-4 |
| **Cardiovascular** | Heart/circulation | MAP, vasopressor use | 0-4 |
| **Neurological** | Brain function | Glasgow Coma Scale | 0-4 |
| **Renal** | Kidney function | Creatinine | 0-4 |

**Total SOFA = sum of all 6 components (0-24).** Higher = sicker.

### What This Notebook Produces

| Output File | Shape | Description |
|------------|-------|-------------|
| `obs_sequences.npy` | Array of (T_i, 14) arrays | 14 observation features per time point per patient |
| `int_sequences.npy` | Array of (T_i, 3) arrays | 3 binary treatment flags per time point |
| `seq_lengths.npy` | (N_patients,) | Number of observations per patient |
| `baseline_sofas.npy` | (N_patients,) | SOFA score at ICU admission |
| `mortality_labels.npy` | (N_patients,) | 0=survived, 1=died |
| `time_points.npy` | Array of float arrays | Actual hours_in values |
| `feature_names.json` | Dict | Names of obs and treatment features |
| `scaler_params.json` | Dict | StandardScaler means and stds (for inverse transform) |

### The 14 Observation Features
```
Index  Feature        Source       Clinical Meaning
0      heart_rate     chartevents  Cardiac stress indicator
1      sbp            chartevents  Systolic blood pressure
2      map            chartevents  Mean arterial pressure (perfusion)
3      temperature    chartevents  Infection/inflammation
4      resp_rate      chartevents  Respiratory distress
5      spo2           chartevents  Blood oxygen saturation
6      lactate        labevents    Tissue hypoperfusion marker
7      wbc            labevents    White blood cell count (infection)
8      platelets      labevents    Coagulation (SOFA component)
9      creatinine     labevents    Kidney function (SOFA component)
10     bilirubin      labevents    Liver function (SOFA component)
11     hemoglobin     labevents    Oxygen carrying capacity
12     sofa_total     computed     Composite severity score (0-24)
13     delta_t        computed     Hours since previous observation
```

## 📌 Cell 1: Configuration

Load the continuous-time data from Step 2 and define which features the HMM will use.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os, time, json, warnings
warnings.filterwarnings('ignore')

# =============================================================
# 🔧 USER CONFIGURATION
# =============================================================
MODEL_DIR = "/Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/03_Model_Ready_Data"

# 🔧 TUNABLE: HMM observation variables (+ delta_t added!)
OBSERVATION_FEATURES = [
    'heart_rate', 'sbp', 'map', 'temperature', 'resp_rate', 'spo2',
    'lactate', 'wbc', 'platelets', 'creatinine', 'bilirubin', 'hemoglobin',
    'sofa_total',
    'delta_t',          # ⚡ Time gap between observations (key for continuous time!)
]

INTERVENTION_FEATURES = [
    'on_antibiotics', 'on_vasopressors', 'on_iv_fluids',
]

MIN_VALID_POINTS = 4    # Minimum observation count (not hours!)

start_time = time.time()
continuous = pd.read_csv(f"{MODEL_DIR}/sepsis_continuous.csv")
print(f"Loaded: {continuous.shape[0]:,}rows × {continuous.shape[1]}columns (continuous)")

Loaded: 284,673rows × 44columns (continuous)


## 📌 Cell 2: SOFA Score Computation

Computes SOFA score for **every observation time point** in the dataset.

Each of the 6 organ-specific scoring functions follows the standard SOFA criteria:
- **Respiratory**: Based on PaO2/FiO2 ratio (gold standard) or SpO2 (fallback when blood gas unavailable)
- **Coagulation**: Platelet count thresholds (< 150K → score 1, < 20K → score 4)
- **Liver**: Bilirubin thresholds (≥ 1.2 → score 1, ≥ 12 → score 4)
- **Cardiovascular**: MAP < 70 → score 1; on vasopressors → score 3
- **Neurological**: Glasgow Coma Scale (GCS) — lower = worse
- **Renal**: Creatinine thresholds (≥ 1.2 → score 1, ≥ 5 → score 4)

**Note**: When a measurement is missing (NaN), the component defaults to 0.
This is a conservative assumption — "if we didn't measure it, assume it's normal."


In [2]:
print("=" * 60)
print("SOFA SCORE COMPUTATION")
print("=" * 60)

# Component 1: Respiration
def sofa_respiration(row):
    po2 = row.get('po2', np.nan)
    fio2 = row.get('fio2', np.nan)
    spo2 = row.get('spo2', np.nan)
    if pd.notna(po2) and pd.notna(fio2) and fio2 > 0:
        fio2_frac = fio2 / 100 if fio2 > 1 else fio2
        ratio = po2 / fio2_frac
        if ratio < 100: return 4
        elif ratio < 200: return 3
        elif ratio < 300: return 2
        elif ratio < 400: return 1
        else: return 0
    elif pd.notna(spo2):
        if spo2 < 90: return 3
        elif spo2 < 95: return 1
        else: return 0
    return 0

def sofa_coagulation(platelets):
    if pd.isna(platelets): return 0
    if platelets < 20: return 4
    elif platelets < 50: return 3
    elif platelets < 100: return 2
    elif platelets < 150: return 1
    return 0

def sofa_liver(bilirubin):
    if pd.isna(bilirubin): return 0
    if bilirubin >= 12: return 4
    elif bilirubin >= 6: return 3
    elif bilirubin >= 2: return 2
    elif bilirubin >= 1.2: return 1
    return 0

def sofa_cardiovascular(row):
    map_val = row.get('map', np.nan)
    on_vaso = row.get('on_vasopressors', 0)
    if on_vaso == 1: return 3
    elif pd.notna(map_val) and map_val < 70: return 1
    return 0

def sofa_cns(row):
    gcs_eye = row.get('gcs_eye', np.nan)
    gcs_verbal = row.get('gcs_verbal', np.nan)
    gcs_motor = row.get('gcs_motor', np.nan)
    valid = [c for c in [gcs_eye, gcs_verbal, gcs_motor] if pd.notna(c)]
    if not valid: return 0
    gcs = sum(valid) + (3 - len(valid))
    if gcs < 6: return 4
    elif gcs < 10: return 3
    elif gcs < 13: return 2
    elif gcs < 15: return 1
    return 0

def sofa_renal(creatinine):
    if pd.isna(creatinine): return 0
    if creatinine >= 5: return 4
    elif creatinine >= 3.5: return 3
    elif creatinine >= 2: return 2
    elif creatinine >= 1.2: return 1
    return 0

continuous['sofa_resp'] = continuous.apply(sofa_respiration, axis=1)
continuous['sofa_coag'] = continuous['platelets'].apply(sofa_coagulation) if 'platelets' in continuous.columns else 0
continuous['sofa_liver'] = continuous['bilirubin'].apply(sofa_liver) if 'bilirubin' in continuous.columns else 0
continuous['sofa_cardio'] = continuous.apply(sofa_cardiovascular, axis=1)
continuous['sofa_cns'] = continuous.apply(sofa_cns, axis=1)
continuous['sofa_renal'] = continuous['creatinine'].apply(sofa_renal) if 'creatinine' in continuous.columns else 0

continuous['sofa_total'] = (
    continuous['sofa_resp'] + continuous['sofa_coag'] + continuous['sofa_liver'] +
    continuous['sofa_cardio'] + continuous['sofa_cns'] + continuous['sofa_renal']
)

print(f"SOFA Score Summary:")
print(f"  Mean:   {continuous['sofa_total'].mean():.2f}")
print(f"  Median: {continuous['sofa_total'].median():.1f}")
print(f"  0-1:    {(continuous['sofa_total'] <= 1).mean():.1%} (Minimal)")
print(f"  2-5:    {((continuous['sofa_total'] >= 2) & (continuous['sofa_total'] <= 5)).mean():.1%} (Mild)")
print(f"  6-9:    {((continuous['sofa_total'] >= 6) & (continuous['sofa_total'] <= 9)).mean():.1%} (Severe)")
print(f"  10+:    {(continuous['sofa_total'] >= 10).mean():.1%} (Shock)")

SOFA SCORE COMPUTATION
SOFA Score Summary:
  Mean:   3.45
  Median: 3.0
  0-1:    35.7% (Minimal)
  2-5:    41.0% (Mild)
  6-9:    17.5% (Severe)
  10+:    5.8% (Shock)


## 📌 Cell 3: Baseline SOFA & Feature Engineering

**Baseline SOFA** = the SOFA score at the patient's **first observation** in the ICU.
This captures "how sick was the patient when they arrived?"

Used in the HMM as `beta_severity × baseline_SOFA` to model that patients who
arrive sicker have inherently different transition dynamics (partially addresses
confounding by indication for vasopressors).

In [3]:
print("\n" + "=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# Baseline SOFA: First observation SOFA for each patient
baseline = continuous.sort_values(['stay_id', 'hours_in']).groupby('stay_id').first()[['sofa_total']].reset_index()
baseline.columns = ['stay_id', 'baseline_sofa']
continuous = continuous.merge(baseline, on='stay_id', how='left')
continuous['baseline_sofa'] = continuous['baseline_sofa'].fillna(continuous['sofa_total'].median())

print(f"  Baseline SOFA: mean={continuous['baseline_sofa'].mean():.2f}")


FEATURE ENGINEERING
  Baseline SOFA: mean=0.67


## 📌 Cell 4: Feature Scaling & Sequence Preparation

### Feature Scaling
All 14 observation features are **standardized** (mean=0, std=1) using `StandardScaler`.
This is critical because:
- Features have wildly different scales (heart rate ~60-120 vs bilirubin ~0.1-12)
- The HMM's Gaussian emission model assumes features are on comparable scales
- Without scaling, high-magnitude features dominate the likelihood

### Sequence Preparation
Each patient's data is packaged into a separate numpy array:
- `obs_sequences[i]`: shape (T_i, 14) — all 14 features at each time point
- `int_sequences[i]`: shape (T_i, 3) — treatment flags at each time point
- `time_points[i]`: shape (T_i,) — actual hours_in values

Patients with fewer than `MIN_VALID_POINTS` (4) observations are excluded.

### Why Object Arrays?
Patients have **different sequence lengths** (19 to 928 observations).
We store sequences as numpy object arrays — each element is a 2D float array
of different length. Padding to equal length happens in Step 5.

In [4]:
print("\n" + "=" * 60)
print("FEATURE SCALING & SEQUENCE PREPARATION")
print("=" * 60)

obs_cols = [c for c in OBSERVATION_FEATURES if c in continuous.columns]
int_cols = [c for c in INTERVENTION_FEATURES if c in continuous.columns]
print(f"  Observation features ({len(obs_cols)}): {obs_cols}")
print(f"  Intervention features ({len(int_cols)}): {int_cols}")

# Missing values: median imputation
for col in obs_cols:
    median_val = continuous[col].median()
    continuous[col] = continuous[col].fillna(median_val)

# StandardScaler
scaler = StandardScaler()
continuous[obs_cols] = scaler.fit_transform(continuous[obs_cols])

scaler_params = {
    'means': dict(zip(obs_cols, scaler.mean_.tolist())),
    'stds': dict(zip(obs_cols, scaler.scale_.tolist()))
}

# Sequence generation
obs_sequences = []
int_sequences = []
seq_lengths = []
stay_ids = []
mortality_labels = []
baseline_sofas = []
time_points = []     # ⚡ Save hours_in for each observation

stay_groups = continuous.groupby('stay_id')
for sid, group in stay_groups:
    group = group.sort_values('hours_in')
    obs = group[obs_cols].values
    interv = group[int_cols].values if int_cols else np.zeros((len(group), 1))
    
    if len(obs) < MIN_VALID_POINTS:
        continue
    
    obs_sequences.append(obs.astype(np.float32))
    int_sequences.append(interv.astype(np.float32))
    seq_lengths.append(len(obs))
    stay_ids.append(sid)
    mortality_labels.append(group['mortality'].iloc[0] if 'mortality' in group.columns else 0)
    baseline_sofas.append(group['baseline_sofa'].iloc[0] if 'baseline_sofa' in group.columns else 0)
    time_points.append(group['hours_in'].values.astype(np.float32))

obs_sequences = np.array(obs_sequences, dtype=object)
int_sequences = np.array(int_sequences, dtype=object)
seq_lengths = np.array(seq_lengths)
stay_ids = np.array(stay_ids)
mortality_labels = np.array(mortality_labels)
baseline_sofas = np.array(baseline_sofas)
time_points = np.array(time_points, dtype=object)

print(f"\n  Patient count: {len(obs_sequences):,}")
print(f"  Sequence length: median={np.median(seq_lengths):.0f}, "
      f"mean={np.mean(seq_lengths):.1f}, max={np.max(seq_lengths)}")
print(f"  Obs dimensions: {obs_sequences[0].shape[1]} (including delta_t)")
print(f"  Treatment dimensions: {int_sequences[0].shape[1]}")
print(f"  Mortality rate: {mortality_labels.mean():.1%}")


FEATURE SCALING & SEQUENCE PREPARATION
  Observation features (14): ['heart_rate', 'sbp', 'map', 'temperature', 'resp_rate', 'spo2', 'lactate', 'wbc', 'platelets', 'creatinine', 'bilirubin', 'hemoglobin', 'sofa_total', 'delta_t']
  Intervention features (3): ['on_antibiotics', 'on_vasopressors', 'on_iv_fluids']

  Patient count: 1,502
  Sequence length: median=170, mean=189.5, max=928
  Obs dimensions: 14 (including delta_t)
  Treatment dimensions: 3
  Mortality rate: 17.2%


## 📌 Cell 5: Save All Output Files

Save everything the HMM needs. The scaler parameters are saved separately so that Step 6 can inverse-transform predictions back to clinical units if needed.

In [5]:
print("\n" + "=" * 60)
print("SAVING")
print("=" * 60)

np.save(f"{MODEL_DIR}/obs_sequences.npy", obs_sequences)
np.save(f"{MODEL_DIR}/int_sequences.npy", int_sequences)
np.save(f"{MODEL_DIR}/seq_lengths.npy", seq_lengths)
np.save(f"{MODEL_DIR}/stay_ids.npy", stay_ids)
np.save(f"{MODEL_DIR}/mortality_labels.npy", mortality_labels)
np.save(f"{MODEL_DIR}/baseline_sofas.npy", baseline_sofas)
np.save(f"{MODEL_DIR}/time_points.npy", time_points)  # ⚡ continuous time

with open(f"{MODEL_DIR}/feature_names.json", 'w') as f:
    json.dump({'obs_features': obs_cols, 'int_features': int_cols}, f, indent=2)

with open(f"{MODEL_DIR}/scaler_params.json", 'w') as f:
    json.dump(scaler_params, f, indent=2)

continuous.to_csv(f"{MODEL_DIR}/sepsis_continuous_with_sofa.csv", index=False)

total_time = time.time() - start_time
print(f"\n✅ SOFA & Feature Engineering Complete! ({total_time:.0f}sec)")
print(f"  Save location: {MODEL_DIR}")
print(f"\n  Output files:")
print(f"    obs_sequences.npy  — Observation sequences ({len(obs_sequences):,} patients, including delta_t)")
print(f"    int_sequences.npy  — Treatment sequences")
print(f"    time_points.npy    — Actual time of each obs (hours_in)")
print(f"    feature_names.json — variable names")
print(f"\n🔜 Next: 05_Bayesian_HMM_ADVI.ipynb Run")


SAVING

✅ SOFA & Feature Engineering Complete! (11sec)
  Save location: /Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/03_Model_Ready_Data

  Output files:
    obs_sequences.npy  — Observation sequences (1,502 patients, including delta_t)
    int_sequences.npy  — Treatment sequences
    time_points.npy    — Actual time of each obs (hours_in)
    feature_names.json — variable names

🔜 Next: 05_Bayesian_HMM_ADVI.ipynb Run
